<a href="https://colab.research.google.com/github/SahajBiyani/MMxLR/blob/main/context.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Background

In this task I provide you with a trained model, and ask you to investigate what it's doing.

We want to figure out what the model has learned: which features does it represent, what does it do with the others, and what "trick" is it using to perform better than our naive expectation?

# Setup code (imports)

In [ ]:
import pickle
from pathlib import Path
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

# Model & training code

The model is a one-hidden-layer MLP without biases and without a skip connection. The architecture is:

```
    x ∈ R^(batch × n_features)
         |
         v
    +----------+
    |   W_in   |  (n_neurons × n_features), trainable
    +----------+
         |
         v
    +----------+
    |   ReLU   |
    +----------+
         |
         v
    +----------+
    |  W_out   |  (n_features × n_neurons), trainable
    +----------+
         |
         v
    ŷ ∈ R^(batch × n_features)
```

That is: $\hat{y} = W_{\text{out}} \cdot \text{ReLU}(W_{\text{in}} \cdot x^T)^T$.

Note there is no skip connection. The entire computation goes through the MLP. The target is $y = \text{ReLU}(x)$.

## Data generation

Each input $x$ is a sparse vector: each of the 100 features is independently active with probability $p = 0.02$. When active, the value is drawn from $\text{Uniform}(-1, 1)$. Inactive features are 0.

## Model and training code

The model code below defines the architecture and training procedure. You don't need to read the training code in detail.

In [ ]:
class SimpleMLP(nn.Module):
    def __init__(self, n_features, n_neurons):
        super().__init__()
        self.W_in = nn.Parameter(torch.randn(n_neurons, n_features) * 0.01)
        self.W_out = nn.Parameter(torch.randn(n_features, n_neurons) * 0.01)

    def forward(self, x):
        return (self.W_out @ torch.relu(self.W_in @ x.T)).T


def generate_batch(batch_size, n_features, p, device=DEVICE):
    mask = (torch.rand(batch_size, n_features, device=device) < p).float()
    values = torch.rand(batch_size, n_features, device=device) * 2 - 1
    x = mask * values
    y = torch.relu(x)
    return x, y


def train_model(n_features, n_neurons, p, loss_exp, n_batches=1000):
    model = SimpleMLP(n_features, n_neurons).to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=0.003)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_batches)

    for step in range(n_batches):
        x, y = generate_batch(2048, n_features, p)
        y_hat = model(x)
        loss = (torch.abs(y_hat - y) ** loss_exp).mean()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()

        if step % 200 == 0 or step == n_batches - 1:
            print(f"Step {step:5d} | L{loss_exp} loss: {loss.item():.2e}")

    return model


def evaluate_per_feature(model, n_features, p):
    """Compute per-feature MSE on active features."""
    model.eval()
    per_feat = torch.zeros(n_features, device=DEVICE)
    count = torch.zeros(n_features, device=DEVICE)
    with torch.no_grad():
        for _ in range(100):
            x, y = generate_batch(2048, n_features, p)
            active = (x != 0).float()
            err = ((model(x) - y) ** 2 * active).sum(dim=0)
            per_feat += err
            count += active.sum(dim=0)
    per_feat = (per_feat / count.clamp(min=1)).cpu().numpy()
    model.train()
    return per_feat

# Training

The following cell trains the model (should take ~20 seconds on CPU, so no GPU needed). It should automatically load the model the 2nd time you run this cell, otherwise adjust the code below.

In [ ]:
N_FEATURES = 100
N_NEURONS = 10
P = 0.02
LOSS_EXP = 4

model_path = Path("model.pkl")
if not model_path.exists():
  torch.manual_seed(42)
  np.random.seed(42)
  model = train_model(N_FEATURES, N_NEURONS, P, LOSS_EXP)
  with open(model_path, "wb") as f:
    pickle.dump(model, f)
else:
  with open(model_path, "rb") as f:
    model = pickle.load(f)

# Task (part 1)

This model has only 10 neurons, but was trained on 100 input features. We want to see what the model has learned.

In this exercise I want to test how well you can iterate on results and find good ways to extract and present information. The suggestions below are what I might say during a mentoring call. You don't have to stick to them exactly, and you certainly don't have to stop there!

> I think we should start by measuring the loss per feature, can you plot this? And can you plot the input-output response for individual features (where input feature = output feature), in isolation (only that feature non-zero).

# Task (part 2)

In [ ]:
class NaiveSolution(nn.Module):
    """Dedicate each of the 10 neurons to one feature.

    Perfectly computes ReLU for 10 features, outputs 0 for the other 90.
    """
    def __init__(self, n_features, n_neurons):
        super().__init__()
        # W_in: each neuron reads from exactly one feature
        W_in = torch.zeros(n_neurons, n_features)
        for i in range(n_neurons):
            W_in[i, i] = 1.0
        # W_out: each neuron writes to exactly one feature
        W_out = torch.zeros(n_features, n_neurons)
        for i in range(n_neurons):
            W_out[i, i] = 1.0
        self.register_buffer("W_in", W_in)
        self.register_buffer("W_out", W_out)

    def forward(self, x):
        return (self.W_out @ torch.relu(self.W_in @ x.T)).T


naive_model = NaiveSolution(N_FEATURES, N_NEURONS).to(DEVICE)
naive_per_feat = evaluate_per_feature(naive_model, N_FEATURES, P)
print(f"Naive model: mean per-feature MSE = {naive_per_feat.mean():.4f}")
print(f"Trained model: mean per-feature MSE = {evaluate_per_feature(model, N_FEATURES, P).mean():.4f}")

The trained model achieves a better loss than the naive solution (which perfectly computes ReLU for 10 features and outputs 0 for the rest). How does it do that? What is it doing?

> I would generally like to check how the model loss scales with the number of active features. During training, each feature is active with $p = 0.02$ (so ~2 active on average), but we can vary this. Can you plot the "average loss per active feature" as a function of the number of active features?

> And after that, maybe check if there's any pattern in the input-output response of the "cross-terms", i.e. where input feature $\neq$ output feature?